# 🖥️ Laptop Price Prediction
## Step 2: Exploratory Data Analysis (EDA)
**Dataset:** [Laptop Price - Kaggle](https://www.kaggle.com/datasets/muhammetvarl/laptop-price)  
**Student:** Shivakumar | B.Tech CSE | VEMU Institute of Technology

---
## 📦 0. Install & Import Libraries

In [ ]:
!pip install kaggle -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

print("✅ Libraries imported successfully")


---
## 📂 1. Load Dataset
> Upload `laptop_price.csv` manually OR use Kaggle API below.

In [ ]:
# ── Option A: Upload manually ──
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0], encoding='latin-1')

# ── Option B: Kaggle API ──
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d muhammetvarl/laptop-price --unzip

# ── Option C: Direct load (after upload) ──
df = pd.read_csv('laptop_price.csv', encoding='latin-1')

print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


---
## 📐 2. Dataset Shape & Structure

In [ ]:
print("=" * 50)
print(f"  Rows    : {df.shape[0]}")
print(f"  Columns : {df.shape[1]}")
print("=" * 50)
print("\nColumn Names:")
for col in df.columns:
    print(f"  • {col}")


---
## 🔢 3. Data Types

In [ ]:
dtype_df = pd.DataFrame({
    'Column': df.columns,
    'Dtype': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values,
    'Sample Value': [str(df[c].dropna().iloc[0]) if df[c].notnull().any() else 'N/A' for c in df.columns]
})
print(dtype_df.to_string(index=False))


In [ ]:
# Visual: dtype distribution
dtype_counts = df.dtypes.astype(str).value_counts()
fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.barh(dtype_counts.index, dtype_counts.values, color=['#2E5D9E','#C55A11','#27AE60'])
ax.bar_label(bars, padding=4, fontsize=10, fontweight='bold')
ax.set_title('Column Data Types Distribution', fontweight='bold')
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig('eda_01_dtypes.png', bbox_inches='tight')
plt.show()
print("📊 Plot saved: eda_01_dtypes.png")


---
## ❓ 4. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ No missing values found in the dataset!")
else:
    print(missing_df)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(missing_df.index, missing_df['Missing %'], color='#E74C3C')
    ax.set_title('Missing Values by Column (%)', fontweight='bold')
    ax.set_xlabel('Missing %')
    for i, v in enumerate(missing_df['Missing %']):
        ax.text(v + 0.2, i, f'{v}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('eda_02_missing.png', bbox_inches='tight')
    plt.show()


---
## 🔁 5. Duplicate Records

In [ ]:
n_dup = df.duplicated().sum()
print(f"Total duplicate rows : {n_dup}")
print(f"Unique rows          : {len(df) - n_dup}")
print(f"Duplication rate     : {n_dup/len(df)*100:.2f}%")

if n_dup > 0:
    print("\nSample duplicate rows:")
    display(df[df.duplicated(keep=False)].head(4))
else:
    print("✅ No duplicate records found.")


---
## 📊 6. Statistical Summary

In [ ]:
print("─── Numerical Columns ───")
display(df.describe().T.round(2))


In [ ]:
print("─── Categorical Columns ───")
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"\n{col} ({df[col].nunique()} unique values):")
    print(df[col].value_counts().head(8).to_string())


---
## 🎯 7. Target Variable Distribution — `Price_euros`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram
axes[0].hist(df['Price_euros'], bins=40, color='#2E5D9E', edgecolor='white', alpha=0.85)
axes[0].set_title('Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price (€)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['Price_euros'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: €{df['Price_euros'].mean():.0f}")
axes[0].axvline(df['Price_euros'].median(), color='orange', linestyle='--', linewidth=1.5, label=f"Median: €{df['Price_euros'].median():.0f}")
axes[0].legend(fontsize=8)

# Log-transformed histogram
axes[1].hist(np.log1p(df['Price_euros']), bins=40, color='#27AE60', edgecolor='white', alpha=0.85)
axes[1].set_title('Log(Price) Distribution', fontweight='bold')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Frequency')

# Box plot
axes[2].boxplot(df['Price_euros'], vert=True, patch_artist=True,
    boxprops=dict(facecolor='#C55A11', alpha=0.6),
    medianprops=dict(color='black', linewidth=2))
axes[2].set_title('Price Boxplot', fontweight='bold')
axes[2].set_ylabel('Price (€)')

plt.suptitle('Target Variable: Price_euros Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_03_target_dist.png', bbox_inches='tight')
plt.show()

print(f"\nPrice Statistics:")
print(f"  Min    : €{df['Price_euros'].min():.2f}")
print(f"  Max    : €{df['Price_euros'].max():.2f}")
print(f"  Mean   : €{df['Price_euros'].mean():.2f}")
print(f"  Median : €{df['Price_euros'].median():.2f}")
print(f"  Std    : €{df['Price_euros'].std():.2f}")
print(f"  Skew   : {df['Price_euros'].skew():.3f}  ({'right-skewed' if df['Price_euros'].skew() > 0.5 else 'approx. normal'})")


---
## 🏷️ 8. Categorical Feature Distributions

In [ ]:
cat_cols_plot = ['Company', 'TypeName', 'OpSys']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#2E5D9E', '#27AE60', '#C55A11']
for ax, col, color in zip(axes, cat_cols_plot, colors):
    vc = df[col].value_counts()
    bars = ax.barh(vc.index, vc.values, color=color, alpha=0.85, edgecolor='white')
    ax.bar_label(bars, padding=3, fontsize=8)
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel('Count')
    ax.invert_yaxis()

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_04_categorical.png', bbox_inches='tight')
plt.show()


---
## 💶 9. Price by Categorical Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['Company', 'TypeName', 'OpSys']):
    order = df.groupby(col)['Price_euros'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, y=col, x='Price_euros', order=order, ax=ax,
                palette='Blues_d', linewidth=0.8)
    ax.set_title(f'Price by {col}', fontweight='bold')
    ax.set_xlabel('Price (€)')
    ax.set_ylabel('')

plt.suptitle('Price Distribution by Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_05_price_by_cat.png', bbox_inches='tight')
plt.show()


---
## 🧮 10. RAM Distribution & Price Relationship

In [ ]:
# Parse RAM
df['Ram_GB'] = df['Ram'].str.replace('GB','').astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# RAM value counts
vc = df['Ram_GB'].value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color='#2E5D9E', edgecolor='white', alpha=0.85)
axes[0].set_title('RAM Distribution', fontweight='bold')
axes[0].set_xlabel('RAM (GB)')
axes[0].set_ylabel('Count')
for i, (x, v) in enumerate(zip(vc.index, vc.values)):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=8)

# RAM vs Price
ram_price = df.groupby('Ram_GB')['Price_euros'].median().reset_index()
axes[1].plot(ram_price['Ram_GB'], ram_price['Price_euros'], 'o-', color='#C55A11', linewidth=2, markersize=7)
axes[1].fill_between(ram_price['Ram_GB'], ram_price['Price_euros'], alpha=0.15, color='#C55A11')
axes[1].set_title('Median Price by RAM', fontweight='bold')
axes[1].set_xlabel('RAM (GB)')
axes[1].set_ylabel('Median Price (€)')

plt.tight_layout()
plt.savefig('eda_06_ram.png', bbox_inches='tight')
plt.show()


---
## 🖥️ 11. Screen Size Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Inches distribution
axes[0].hist(df['Inches'].astype(float), bins=20, color='#27AE60', edgecolor='white', alpha=0.85)
axes[0].set_title('Screen Size Distribution', fontweight='bold')
axes[0].set_xlabel('Screen Size (inches)')
axes[0].set_ylabel('Count')

# Inches vs Price scatter
axes[1].scatter(df['Inches'].astype(float), df['Price_euros'],
    alpha=0.4, color='#2E5D9E', s=20, edgecolors='none')
# Trend line
z = np.polyfit(df['Inches'].astype(float), df['Price_euros'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Inches'].astype(float).min(), df['Inches'].astype(float).max(), 100)
axes[1].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
axes[1].set_title('Screen Size vs Price', fontweight='bold')
axes[1].set_xlabel('Screen Size (inches)')
axes[1].set_ylabel('Price (€)')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_07_screen.png', bbox_inches='tight')
plt.show()


---
## ⚙️ 12. CPU Brand Analysis

In [ ]:
# Extract CPU brand
df['cpu_brand'] = df['Cpu'].apply(lambda x: x.split()[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# CPU brand count
vc = df['cpu_brand'].value_counts()
axes[0].bar(vc.index, vc.values, color=['#2E5D9E','#C55A11','#27AE60','#8E44AD'][:len(vc)],
    edgecolor='white', alpha=0.9)
axes[0].set_title('CPU Brand Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=9, fontweight='bold')

# CPU brand vs Price
sns.boxplot(data=df, x='cpu_brand', y='Price_euros', ax=axes[1], palette='Set2', linewidth=0.8)
axes[1].set_title('Price by CPU Brand', fontweight='bold')
axes[1].set_xlabel('CPU Brand')
axes[1].set_ylabel('Price (€)')

plt.tight_layout()
plt.savefig('eda_08_cpu.png', bbox_inches='tight')
plt.show()


---
## 🎮 13. GPU Brand Analysis

In [ ]:
df['gpu_brand'] = df['Gpu'].apply(lambda x: x.split()[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

vc = df['gpu_brand'].value_counts()
axes[0].bar(vc.index, vc.values, color=['#2E5D9E','#C55A11','#27AE60','#E74C3C'][:len(vc)],
    edgecolor='white', alpha=0.9)
axes[0].set_title('GPU Brand Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontsize=9, fontweight='bold')

sns.boxplot(data=df, x='gpu_brand', y='Price_euros', ax=axes[1], palette='coolwarm', linewidth=0.8)
axes[1].set_title('Price by GPU Brand', fontweight='bold')
axes[1].set_xlabel('GPU Brand')
axes[1].set_ylabel('Price (€)')

plt.tight_layout()
plt.savefig('eda_09_gpu.png', bbox_inches='tight')
plt.show()


---
## 💾 14. Storage Type Analysis

In [ ]:
def get_storage_type(mem):
    mem = str(mem)
    has_ssd = 'SSD' in mem or 'Flash' in mem or 'eMMC' in mem
    has_hdd = 'HDD' in mem
    if has_ssd and has_hdd:
        return 'Hybrid (SSD+HDD)'
    elif has_ssd:
        return 'SSD Only'
    elif has_hdd:
        return 'HDD Only'
    else:
        return 'Other'

df['storage_type'] = df['Memory'].apply(get_storage_type)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

vc = df['storage_type'].value_counts()
colors_pie = ['#2E5D9E', '#27AE60', '#C55A11', '#8E44AD']
axes[0].pie(vc.values, labels=vc.index, autopct='%1.1f%%',
    colors=colors_pie[:len(vc)], startangle=140, textprops={'fontsize': 9})
axes[0].set_title('Storage Type Distribution', fontweight='bold')

sns.boxplot(data=df, x='storage_type', y='Price_euros', ax=axes[1], palette='Set3', linewidth=0.8)
axes[1].set_title('Price by Storage Type', fontweight='bold')
axes[1].set_xlabel('Storage Type')
axes[1].set_ylabel('Price (€)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('eda_10_storage.png', bbox_inches='tight')
plt.show()


---
## ⚖️ 15. Weight Analysis

In [ ]:
df['Weight_kg'] = df['Weight'].str.replace('kg','').astype(float)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['Weight_kg'], bins=25, color='#8E44AD', edgecolor='white', alpha=0.85)
axes[0].set_title('Weight Distribution', fontweight='bold')
axes[0].set_xlabel('Weight (kg)')
axes[0].set_ylabel('Count')

axes[1].scatter(df['Weight_kg'], df['Price_euros'], alpha=0.4, color='#8E44AD', s=20)
z = np.polyfit(df['Weight_kg'], df['Price_euros'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Weight_kg'].min(), df['Weight_kg'].max(), 100)
axes[1].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
axes[1].set_title('Weight vs Price', fontweight='bold')
axes[1].set_xlabel('Weight (kg)')
axes[1].set_ylabel('Price (€)')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_11_weight.png', bbox_inches='tight')
plt.show()


---
## 🔗 16. Correlation Analysis

In [ ]:
# Build numerical df for correlation
num_df = pd.DataFrame({
    'Price_euros': df['Price_euros'],
    'Ram_GB': df['Ram_GB'],
    'Inches': df['Inches'].astype(float),
    'Weight_kg': df['Weight_kg'],
})

corr = num_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=axes[0],
    annot_kws={'size': 11, 'weight': 'bold'})
axes[0].set_title('Correlation Matrix (Numerical Features)', fontweight='bold')

# Bar: correlation with Price
price_corr = corr['Price_euros'].drop('Price_euros').sort_values()
colors = ['#E74C3C' if v < 0 else '#27AE60' for v in price_corr]
bars = axes[1].barh(price_corr.index, price_corr.values, color=colors, edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Feature Correlation with Price', fontweight='bold')
axes[1].set_xlabel('Pearson Correlation Coefficient')
for bar, val in zip(bars, price_corr.values):
    axes[1].text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
        f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('eda_12_correlation.png', bbox_inches='tight')
plt.show()

print("\nCorrelation with Price_euros:")
print(corr['Price_euros'].sort_values(ascending=False).to_string())


---
## 📦 17. Outlier Analysis (IQR Method)

In [ ]:
def iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    return Q1, Q3, IQR, lower, upper, len(outliers)

num_cols = ['Price_euros', 'Ram_GB', 'Inches', 'Weight_kg']
print(f"{'Column':<15} {'Q1':>8} {'Q3':>8} {'IQR':>8} {'Lower':>8} {'Upper':>8} {'Outliers':>10}")
print("-" * 70)
for col in num_cols:
    Q1, Q3, IQR, lower, upper, n = iqr_outliers(num_df[col])
    print(f"{col:<15} {Q1:>8.2f} {Q3:>8.2f} {IQR:>8.2f} {lower:>8.2f} {upper:>8.2f} {n:>10}")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
titles = ['Price (€)', 'RAM (GB)', 'Screen Size (inches)', 'Weight (kg)']

for ax, col, title in zip(axes, num_cols, titles):
    bp = ax.boxplot(num_df[col], patch_artist=True, vert=True,
        boxprops=dict(facecolor='#2E5D9E', alpha=0.5),
        medianprops=dict(color='#C55A11', linewidth=2.5),
        whiskerprops=dict(linewidth=1.5),
        flierprops=dict(marker='o', markerfacecolor='#E74C3C', markersize=4, alpha=0.6))
    ax.set_title(f'{title}\nOutlier Box Plot', fontweight='bold')
    ax.set_ylabel(title)

plt.suptitle('Outlier Analysis — Box Plots (IQR Method)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_13_outliers.png', bbox_inches='tight')
plt.show()


---
## 🔍 18. Pairplot — Numerical Features

In [ ]:
pp = sns.pairplot(num_df, diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15, 'color': '#2E5D9E'})
pp.fig.suptitle('Pairplot: Numerical Features', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('eda_14_pairplot.png', bbox_inches='tight')
plt.show()


---
## 🏢 19. Company × Laptop Type Heatmap (Avg Price)

In [ ]:
pivot = df.pivot_table(values='Price_euros', index='Company', columns='TypeName', aggfunc='mean')

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, annot_kws={'size': 8},
    cbar_kws={'label': 'Avg Price (€)'})
ax.set_title('Average Price (€) — Company × Laptop Type', fontweight='bold', fontsize=13)
ax.set_xlabel('Laptop Type')
ax.set_ylabel('Company')
plt.tight_layout()
plt.savefig('eda_15_heatmap_company_type.png', bbox_inches='tight')
plt.show()


---
## ✅ 20. EDA Summary & Key Findings

In [ ]:
print("=" * 60)
print("        EDA SUMMARY — LAPTOP PRICE PREDICTION")
print("=" * 60)

findings = [
    ("Dataset Shape",        f"{df.shape[0]} rows × {df.shape[1]} columns"),
    ("Missing Values",       "None detected (clean dataset)"),
    ("Duplicates",           f"{df.duplicated().sum()} duplicate rows"),
    ("Price Range",          f"€{df['Price_euros'].min():.0f} — €{df['Price_euros'].max():.0f}"),
    ("Price Mean",           f"€{df['Price_euros'].mean():.0f}"),
    ("Price Median",         f"€{df['Price_euros'].median():.0f}"),
    ("Price Skewness",       f"{df['Price_euros'].skew():.3f} (right-skewed → log transform needed)"),
    ("Top Brand",            df['Company'].value_counts().index[0]),
    ("Most Common Type",     df['TypeName'].value_counts().index[0]),
    ("Most Common RAM",      df['Ram_GB'].value_counts().index[0].astype(str) + "GB"),
    ("Most Common Storage",  df['storage_type'].value_counts().index[0]),
    ("CPU Brand Split",      df['cpu_brand'].value_counts().to_dict()),
    ("GPU Brand Split",      df['gpu_brand'].value_counts().to_dict()),
    ("Key Correlation",      "RAM has highest positive correlation with Price"),
    ("Outliers in Price",    f"{iqr_outliers(df['Price_euros'])[5]} outliers detected (luxury laptops)"),
]

for key, val in findings:
    print(f"  {key:<25}: {val}")

print("=" * 60)
print("\n📌 Next Step → Data Cleaning (Step 3)")
